# Clinical Question Answering (RAG + GPT-4) — MIMIC-III

This notebook runs an end-to-end **Retrieval-Augmented Generation** clinical QA system on the
MIMIC-III data committed to this repository. It retrieves relevant clinical text with
Sentence-BERT + FAISS and generates cited answers with OpenAI GPT-4.

## ⚠️ Where to put your OpenAI API key

You have **two options** (the notebook tries them in this order):

1. **Colab Secrets (recommended):** click the **🔑 key icon** in the left sidebar of Colab →
   **Add new secret** → Name it exactly `OPENAI_API_KEY`, paste your key as the value, and
   toggle **Notebook access** on. Then just run the cells.
2. **Type it when prompted:** if no secret is found, the **API-key cell below** will show a
   password box — paste your key there and press Enter.

> Your key is `sk-...` from <https://platform.openai.com/api-keys>. Never hard-code it into a cell.

## Note on the data

The original project used `NOTEEVENTS.csv` (free-text discharge summaries), which is **not**
part of the MIMIC-III files in this repo. This notebook therefore builds an equivalent
retrievable **clinical-text corpus from the structured tables that are present**
(`ADMISSIONS`, `DIAGNOSES_ICD`, `PROCEDURES_ICD`, `PATIENTS`, plus the ICD dictionaries).
If you later add `NOTEEVENTS.csv`, set `USE_NOTEEVENTS = True` in the corpus cell to use it instead.

## 1. Install dependencies

In [ ]:
# Run once per Colab runtime
!pip -q install openai --upgrade
!pip -q install faiss-cpu sentence-transformers nltk pandas numpy gradio bert-score rouge-score

## 2. Get the data (clone this repository)
All CSVs live in the repo, so no Google Drive is needed.

In [ ]:
import os

DATA_DIR = "Clinical-Question-Answering-Model-using-MIMIC-III-LLMs"
if not os.path.isdir(DATA_DIR):
    !git clone --depth 1 https://github.com/sriya19/Clinical-Question-Answering-Model-using-MIMIC-III-LLMs.git {DATA_DIR}
print("Data files available:")
print([f for f in os.listdir(DATA_DIR) if f.endswith(('.csv', '.csv.gz', '.json'))])

## 3. Set your OpenAI API key
See the instructions at the top of the notebook. Run this cell and either it loads the key
from Colab Secrets automatically, or it prompts you to paste it.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("API key loaded from Colab secrets.")
except Exception:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (sk-...): ")
    print("API key set via prompt.")

assert os.environ.get("OPENAI_API_KEY", "").startswith("sk-"), \
    "No valid OpenAI key found - add OPENAI_API_KEY to Colab Secrets or paste it when prompted."


## 4. Build the clinical-text corpus from the structured data
Each hospital admission is turned into a short clinical narrative (diagnoses, procedures,
disposition). These narratives are what the retriever searches over.

In [ ]:
import pandas as pd

# If you add NOTEEVENTS.csv to the repo, flip this to True to use the real notes instead.
USE_NOTEEVENTS = False
MAX_ADMISSIONS = 3000   # raise for more coverage (slower); full set is ~59k admissions

def load_table(name, **kw):
    for cand in (f"{DATA_DIR}/{name}.csv", f"{DATA_DIR}/{name}.csv.gz"):
        if os.path.exists(cand):
            return pd.read_csv(cand, **kw)
    raise FileNotFoundError(name)

def build_corpus_from_structured(max_admissions=MAX_ADMISSIONS):
    admissions = load_table("ADMISSIONS")
    patients   = load_table("PATIENTS")
    diagnoses  = load_table("DIAGNOSES_ICD")
    d_diag     = load_table("D_ICD_DIAGNOSES")
    procs      = load_table("PROCEDURES_ICD")
    d_proc     = load_table("D_ICD_PROCEDURES")

    diag_titles = dict(zip(d_diag["ICD9_CODE"].astype(str), d_diag["LONG_TITLE"]))
    proc_titles = dict(zip(d_proc["ICD9_CODE"].astype(str), d_proc["LONG_TITLE"]))
    diagnoses["TITLE"] = diagnoses["ICD9_CODE"].astype(str).map(diag_titles)
    procs["TITLE"]     = procs["ICD9_CODE"].astype(str).map(proc_titles)

    diag_by_hadm = (diagnoses.dropna(subset=["TITLE"]).sort_values("SEQ_NUM")
                    .groupby("HADM_ID")["TITLE"].apply(list).to_dict())
    proc_by_hadm = (procs.dropna(subset=["TITLE"]).sort_values("SEQ_NUM")
                    .groupby("HADM_ID")["TITLE"].apply(list).to_dict())
    gender_by_subj = dict(zip(patients["SUBJECT_ID"], patients["GENDER"]))

    notes = []
    for _, row in admissions.head(max_admissions).iterrows():
        hadm, subj = row["HADM_ID"], row["SUBJECT_ID"]
        gender = {"M": "male", "F": "female"}.get(gender_by_subj.get(subj, ""), "patient")
        parts = [f"Admission {hadm}: A {gender} patient (subject {subj}) was admitted on "
                 f"{row.get('ADMITTIME','an unknown date')} via "
                 f"{str(row.get('ADMISSION_TYPE','')).lower()} admission from "
                 f"{row.get('ADMISSION_LOCATION','an unknown location')}."]
        if pd.notna(row.get("DIAGNOSIS")):
            parts.append(f"The presenting complaint was {str(row['DIAGNOSIS']).lower()}.")
        if diag_by_hadm.get(hadm):
            parts.append("Documented diagnoses include: " + "; ".join(diag_by_hadm[hadm][:8]) + ".")
        if proc_by_hadm.get(hadm):
            parts.append("Procedures performed include: " + "; ".join(proc_by_hadm[hadm][:8]) + ".")
        if pd.notna(row.get("DISCHARGE_LOCATION")):
            parts.append(f"The patient was discharged to {str(row['DISCHARGE_LOCATION']).lower()} "
                         f"on {row.get('DISCHTIME','an unknown date')}.")
        if row.get("HOSPITAL_EXPIRE_FLAG", 0) == 1:
            parts.append("The patient expired during this hospital admission.")
        notes.append(" ".join(parts))
    return notes

if USE_NOTEEVENTS and os.path.exists(f"{DATA_DIR}/NOTEEVENTS.csv"):
    clinical_notes = (pd.read_csv(f"{DATA_DIR}/NOTEEVENTS.csv", low_memory=False)
                      ["TEXT"].dropna().tolist()[:MAX_ADMISSIONS])
    print(f"Loaded {len(clinical_notes)} free-text clinical notes from NOTEEVENTS.csv")
else:
    clinical_notes = build_corpus_from_structured()
    print(f"Built {len(clinical_notes)} clinical narratives from structured MIMIC-III tables.")

print("\nExample note:\n", clinical_notes[0][:500])

## 5. Build the retriever (Sentence-BERT + FAISS) and the GPT-4 answerer

In [ ]:
import numpy as np, faiss, nltk
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from nltk.tokenize import sent_tokenize

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

openai_client = OpenAI()
retrieval_model = SentenceTransformer("all-MiniLM-L6-v2")

MAX_SENTENCES = 15000

def split_into_sentences(notes, max_sentences=MAX_SENTENCES):
    sents = []
    for note in notes:
        sents.extend(sent_tokenize(note.strip()))
        if len(sents) >= max_sentences:
            break
    return sents[:max_sentences]

def build_faiss_index(sentences):
    emb = retrieval_model.encode(sentences, convert_to_numpy=True,
                                 batch_size=64, show_progress_bar=True)
    index = faiss.IndexFlatL2(emb.shape[1])
    index.add(emb.astype(np.float32))
    return index

def retrieve(query, sentences, index, top_k=8):
    q = retrieval_model.encode([query], convert_to_numpy=True).astype(np.float32)
    _, idx = index.search(q, top_k)
    return [sentences[i] for i in idx[0]]

def generate_answer(clinician_q, patient_q, top_sentences):
    top_sentences = [s.strip() for s in top_sentences if len(s.strip()) > 10]
    context = "\n".join(f"{i+1}: {s}" for i, s in enumerate(top_sentences))
    prompt = (
        "You are a clinical assistant. Using only the numbered sentences from the clinical "
        "record below, write a professional answer that explains the reason for the treatment "
        "or event. Cite evidence with numbered references like (1), (2). Limit to 75 words.\n\n"
        f"Context:\n{context}\n\nPatient Question: {patient_q}\n"
        f"Clinician Question: {clinician_q}\n\nAnswer:")
    resp = openai_client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "system", "content": "You generate concise, cited clinical explanations."},
                  {"role": "user", "content": prompt}],
        temperature=0.3, max_tokens=200)
    ans = resp.choices[0].message.content.strip()
    words = ans.split()
    return " ".join(words[:75]) + ("..." if len(words) > 75 else "")

# Build the index once
all_sentences = split_into_sentences(clinical_notes)
print(f"Indexing {len(all_sentences)} sentences ...")
index = build_faiss_index(all_sentences)
print("FAISS index ready.")

## 6. Try it (a sample question)
Runs end-to-end so you can confirm the model works before launching the UI.

In [ ]:
patient_q   = "I was admitted for chest pain. What heart procedure was done and why?"
clinician_q = "What coronary procedure was performed and what was the indication?"

top_sentences = retrieve(patient_q + " " + clinician_q, all_sentences, index, top_k=8)
answer = generate_answer(clinician_q, patient_q, top_sentences)

print("Top retrieved sentences:")
for i, s in enumerate(top_sentences, 1):
    print(f"({i}) {s}")
print("\nGenerated answer:\n", answer)

## 7. Interactive Gradio app

In [ ]:
import gradio as gr

def qa_ui(patient_q, clinician_q):
    tops = retrieve(patient_q + " " + clinician_q, all_sentences, index, top_k=8)
    ans = generate_answer(clinician_q, patient_q, tops)
    return "\n".join(f"({i+1}) {s}" for i, s in enumerate(tops)), ans

demo = gr.Interface(
    fn=qa_ui,
    inputs=[gr.Textbox(label="Patient Question"), gr.Textbox(label="Clinician Question")],
    outputs=[gr.Textbox(label="Top Retrieved Sentences"), gr.Textbox(label="Generated Answer (cited)")],
    title="Clinical QA Assistant (RAG + GPT-4)",
    description="Retrieves relevant clinical text from MIMIC-III and generates a cited answer.")
demo.launch(share=True)

## 8. Evaluation (BERTScore + ROUGE) against `test.final.json`

In [ ]:
import json, torch
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from sentence_transformers import util

with open(f"{DATA_DIR}/test.final.json") as f:
    test_data = json.load(f)

def flatten_qa_pairs(test_data):
    out = []
    for art in test_data["data"]:
        for para in art["paragraphs"]:
            for qa in para["qas"]:
                if qa.get("answers"):
                    out.append({"id": qa["id"], "question": qa["question"],
                                "answer": qa["answers"][0]["text"]})
    return out

all_gold = flatten_qa_pairs(test_data)

def most_similar_gold(user_q):
    qs = [g["question"] for g in all_gold]
    ge = retrieval_model.encode(qs, convert_to_tensor=True)
    ue = retrieval_model.encode(user_q, convert_to_tensor=True)
    sims = util.pytorch_cos_sim(ue, ge)[0]
    return all_gold[int(torch.topk(sims, 1).indices[0])]

matched = most_similar_gold(patient_q + " " + clinician_q)
gold_answer = matched["answer"]
print("Closest gold QA id:", matched["id"])

P, R, F1 = bert_score([answer], [gold_answer], lang="en", verbose=False)
print("\n=== Factuality (vs gold answer) ===")
print(f"BERTScore P/R/F1: {P.item():.4f} / {R.item():.4f} / {F1.item():.4f}")

ref = " ".join(top_sentences)
rs = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True).score(ref, answer)
Pr, Rr, F1r = bert_score([answer], [ref], lang="en", verbose=False)
print("\n=== Relevance (vs retrieved context) ===")
print(f"ROUGE-1 F1: {rs['rouge1'].fmeasure:.4f}  ROUGE-L F1: {rs['rougeL'].fmeasure:.4f}  "
      f"BERTScore F1: {F1r.item():.4f}")

print("\nNote: gold answers in test.final.json describe free-text discharge-summary content, "
      "so factuality scores are only meaningful once NOTEEVENTS.csv is added (USE_NOTEEVENTS=True).")